# Seed repeats: 4 vs 8 trainable blocks, without the critic (NYUv2)

This notebook follows the project's seed protocol. The protocol is: 3 seeds for the final model, 1 seed for ablations, and extra seeds only for ablations that drive a decision and fall inside the noise.

Two arms on NYUv2, both without the critic, 3 seeds each: **4 blocks** (E2 of notebook 04 = seed 42, plus seeds 43 and 44) and **8 blocks** (seeds 42–44). The 8-block arm (`ckpt/seeds/nc8_s{42,43,44}`) is the **NYU final model**.


## Setup

***NOTE:*** The DINOv2 and DPT code is used straight from the local repositories `./dinov2` and `./DPT`.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "experiments":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [2]:
from mtl_train import *
from repro_common import PAPER   # the papers' reference numbers, defined once (checked against the PDFs)

configure()
print(f"Device: {device}, GPU: {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'none'}")


Device: cuda, GPU: NVIDIA RTX A4000


## Training

In [3]:
def seed_config(blocks: int, seed: int) -> Tuple[dict, str]:
    """configure() overrides and variant name for one arm and seed (critic always off)."""
    if blocks == 4 and seed == 42:
        roots = dict(ckpt_root=ROOT / "ckpt" / "nocritic", results_root=ROOT / "results" / "nocritic")   # the E2 run of notebook 04
    else:
        roots = dict(ckpt_root=ROOT / "ckpt" / "seeds" / f"nc{blocks}_s{seed}", results_root=ROOT / "results" / "seeds" / f"nc{blocks}_s{seed}")
    return dict(use_critic=False, seed=seed, **roots), ("partial_ft" if blocks == 4 else "partial_ft8")


def setup_run(blocks: int, seed: int) -> str:
    overrides, variant = seed_config(blocks, seed)
    configure(**overrides)
    CFG["variants"]["partial_ft8"] = dict(CFG["variants"]["partial_ft"], n_trainable_blocks=8)
    assert CFG["seed"] == seed and not CFG["use_critic"]
    return variant


SEEDS = [42, 43, 44]
TRAIN_ORDER = [(4, 43), (8, 42), (4, 44), (8, 43), (8, 44)]   # (4, 42) is E2, already trained

for blocks, seed in TRAIN_ORDER:
    variant = setup_run(blocks, seed)
    print(f"=== {blocks} blocks, seed {seed} -> {CFG['ckpt_root']}")
    train_run("nyu", variant)


=== 4 blocks, seed 43 -> /home/chrisjf/Documents/ml4cv/ckpt/seeds/nc4_s43
nyu_partial_ft: already trained
=== 8 blocks, seed 42 -> /home/chrisjf/Documents/ml4cv/ckpt/seeds/nc8_s42
nyu_partial_ft8: already trained
=== 4 blocks, seed 44 -> /home/chrisjf/Documents/ml4cv/ckpt/seeds/nc4_s44
nyu_partial_ft: already trained
=== 8 blocks, seed 43 -> /home/chrisjf/Documents/ml4cv/ckpt/seeds/nc8_s43
nyu_partial_ft8: already trained
=== 8 blocks, seed 44 -> /home/chrisjf/Documents/ml4cv/ckpt/seeds/nc8_s44
nyu_partial_ft8: already trained


## Evaluation: per seed, mean ± std, significance
Every run is scored on the full NYUv2 test set (654 images, Eigen crop) with notebook 03's protocols. The single-pass numbers are the result; sliding-window (TTA) numbers are supplementary only. Results are cached in `<results_root>/eval/`.

In [4]:
def eval_seed_run(blocks: int, seed: int) -> Optional[dict]:
    variant = setup_run(blocks, seed)
    out_path = CFG["results_root"] / "eval" / f"{run_name('nyu', variant)}.json"
    cached = load_json(out_path) if out_path.exists() else None
    if cached and "nyu_depth_slide" in cached:
        return cached
    if not (CFG["ckpt_root"] / run_name("nyu", variant) / "final.pt").exists():
        print(f"{blocks} blocks, seed {seed}: not trained")
        return None
    model = load_trained("nyu", variant)
    results = eval_nyu(model, slide=True)
    save_json(results, out_path)
    del model
    torch.cuda.empty_cache()
    return results


METRICS = [("RMSE", "nyu_depth", "rmse"), ("AbsRel", "nyu_depth", "abs_rel"), ("δ1", "nyu_depth", "d1"), ("mIoU", "nyu_seg", "mIoU"),
           ("RMSE +slide (TTA)", "nyu_depth_slide", "rmse"), ("mIoU +slide (TTA)", "nyu_seg_slide", "mIoU")]

per_seed = []
for blocks in (4, 8):
    for seed in SEEDS:
        results = eval_seed_run(blocks, seed)
        if results:
            per_seed.append({"arm": f"{blocks} blocks", "seed": seed, **{name: results[k][m] for name, k, m in METRICS}})
configure()

per_seed = pd.DataFrame(per_seed)
per_seed.to_csv(ROOT / "results" / "compare" / "seeds_nyu_per_run.csv", index=False)
display(per_seed.style.format(precision=4))

summary = []
for name, _, _ in METRICS:
    a = per_seed.loc[per_seed.arm == "4 blocks", name].to_numpy()
    b = per_seed.loc[per_seed.arm == "8 blocks", name].to_numpy()
    row = {"metric": name, "4 blocks mean": a.mean(), "4 blocks std": a.std(ddof=1) if len(a) > 1 else np.nan,
           "8 blocks mean": b.mean(), "8 blocks std": b.std(ddof=1) if len(b) > 1 else np.nan, "Δ (8 − 4)": b.mean() - a.mean()}
    if len(a) > 1 and len(b) > 1:
        pooled = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
        row["t"] = row["Δ (8 − 4)"] / (pooled * np.sqrt(1 / len(a) + 1 / len(b)))
        row["dof"] = len(a) + len(b) - 2
    summary.append(row)
summary = pd.DataFrame(summary).set_index("metric")
summary.to_csv(ROOT / "results" / "compare" / "seeds_nyu_summary.csv")
display(summary.style.format(precision=4))

print(f"References (single runs, as reported): SwinMTL RMSE {PAPER["swinmtl_tab3"]["SwinMTL"][0]:.3f} / mIoU {PAPER["swinmtl_tab3"]["SwinMTL"][1]} (SwinMTL's per-image metric; mIoU here is the standard metric); "
      f"DINOv2 ViT-S/14 (frozen) + DPT RMSE {PAPER["dinov2_tab11_vits14"]["nyu"][2]}, trained on ~24k NYU frames; DPT-Hybrid RMSE {PAPER["dpt_tab2"]["DPT-Hybrid"]['rmse']}, pretrained on ~1.4 M images and fine-tuned on NYUv2; ours: 795 images.")


,arm,seed,RMSE,AbsRel,δ1,mIoU,RMSE +slide (TTA),mIoU +slide (TTA)
0,4 blocks,42,0.4282,0.1223,0.8601,57.3912,0.4174,59.0322
1,4 blocks,43,0.4320,0.1226,0.8603,57.4087,0.4213,58.9673
2,4 blocks,44,0.4290,0.1230,0.8605,57.3727,0.4184,58.8977
3,8 blocks,42,0.4205,0.1215,0.8637,58.2490,0.4103,59.9522
4,8 blocks,43,0.4205,0.1198,0.8663,58.1626,0.4100,59.8285
5,8 blocks,44,0.4183,0.1199,0.8656,57.8754,0.4087,59.5067


,4 blocks mean,4 blocks std,8 blocks mean,8 blocks std,Δ (8 − 4),t,dof
metric,,,,,,,
RMSE,0.4297,0.0020,0.4197,0.0013,-0.0100,-7.3182,4
AbsRel,0.1226,0.0004,0.1204,0.0010,-0.0022,-3.7553,4
δ1,0.8603,0.0002,0.8652,0.0014,0.0049,6.1757,4
mIoU,57.3909,0.0180,58.0957,0.1956,0.7048,6.2143,4
RMSE +slide (TTA),0.4191,0.0020,0.4096,0.0008,-0.0094,-7.4473,4
mIoU +slide (TTA),58.9657,0.0673,59.7625,0.2300,0.7967,5.7596,4


References (single runs, as reported): SwinMTL RMSE 0.518 / mIoU 58.14 (SwinMTL's per-image metric; mIoU here is the standard metric); DINOv2 ViT-S/14 (frozen) + DPT RMSE 0.356, trained on ~24k NYU frames; DPT-Hybrid RMSE 0.357, pretrained on ~1.4 M images and fine-tuned on NYUv2; ours: 795 images.
